<!-- --- -->
<!-- title: "PUE Classification — A Case Study in Time-Series Model Validation" -->
<!-- author: "[00y300](https://github.com/00y300)" -->
<!-- date: "04-14-2026" -->
<!-- format: -->
<!--   html: -->
<!--     toc: true -->
<!--     toc-depth: 3 -->
<!--     code-fold: false -->
<!--     code-tools: true -->
<!--     theme: cosmo -->
<!--     self-contained: true -->
<!-- jupyter: python3 -->
<!-- execute: -->
<!--   warning: false -->
<!--   message: false -->
<!--   echo: true -->
<!-- --- -->
<!---->
<!-- --- -->

## Summary

Two classification tasks on two years of 5-minute interval data center telemetry: a multiclass efficiency-tier classifier and a binary high-load anomaly detector. The headline result isn't the 94% accuracy the Random Forest produced — it's the four-part validation framework I built around it (persistence baselines, walk-forward cross-validation, feature ablation, and operational threshold tuning), which proved that headline number was largely driven by autocorrelation rather than learned signal.

This project is as much about **how to evaluate a time-series classifier** as it is about training one. Every model number is reported alongside the trivial baseline that defines the floor.

**Stack:** Python · scikit-learn · pandas · SQLite 
**Dataset:** ~200K rows of 5-minute building telemetry (cooling load, IT power, weather, derived PUE)

### Key results at a glance

| Finding | Number |
|---|---|
| Persistence baseline accuracy on tier task | **0.972** |
| Random Forest accuracy on tier task | 0.941 |
| Macro-F1 with all features → weather + time only | **0.92 → 0.30** |
| Walk-forward F1 range across 18 monthly windows | **0.18 – 1.00** (median 0.84) |
| False-positive reduction from threshold tuning | **−71.6%** |

---

## Problem

Two supervised tasks on building telemetry:

1. **PUE Efficiency Tier** — multiclass classification into three efficiency bands
2. **High-Consumption Anomaly Flag** — binary classification of top-decile cooling or IT load events

Both reuse the cleaned dataset (`power_data.db`) and the saved regression models (`pue_5min.pkl`, `pue_1h.pkl`, `pue_24h.pkl`) produced by the upstream forecasting notebook, demonstrating model composition across pipelines.

---

## Setup

In [ ]:
# | label: imports

import numpy as np
import pandas as pd
import pickle
import json
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_recall_curve,
    ConfusionMatrixDisplay,
)

plt.rcParams["savefig.dpi"] = 150
plt.rcParams["figure.figsize"] = [10, 6]


class Config:
    """Shared paths with the regression notebook."""

    BASE_DIR = Path("~/Documents/Projects/MachineLearningProjects").expanduser()
    DATASET_DIR = BASE_DIR / "datasets"
    MODEL_DIR = BASE_DIR / "models"
    PLOT_DIR = BASE_DIR / "plots"

    for d in [DATASET_DIR, MODEL_DIR, PLOT_DIR]:
        d.mkdir(parents=True, exist_ok=True)

    TEST_SPLIT_RATIO = 0.2

---

## Load Pre-Built Dataset and Regression Models

The upstream regression pipeline already wrote `power_data.db` (cleaned and filtered) and three model `.pkl` files. Reusing both here keeps the cleaning logic in a single place and demonstrates how to compose models across notebooks.

In [ ]:
# | label: load-data-and-models

# 1. Cleaned data from SQLite
db_path = Config.DATASET_DIR / "power_data.db"
conn = sqlite3.connect(str(db_path))
df = pd.read_sql(
    "SELECT * FROM building_metrics",
    conn,
    index_col="timestamp",
    parse_dates=["timestamp"],
)
conn.close()
df.index = pd.to_datetime(df.index, utc=True)
print(f"Loaded {len(df):,} rows from {df.index.min()} to {df.index.max()}")

# 2. Saved regression models + the feature list they expect
with open(Config.MODEL_DIR / "feature_columns.json") as f:
    regression_features = json.load(f)

loaded_models = {}
for horizon in ["5min", "1h", "24h"]:
    path = Config.MODEL_DIR / f"pue_{horizon}.pkl"
    with open(path, "rb") as f:
        loaded_models[horizon] = pickle.load(f)
    print(
        f"✓ Loaded {horizon} regression model "
        f"(trained on {loaded_models[horizon]['training_window']})"
    )

print(f"\nRegression feature set ({len(regression_features)}):")
for feat in regression_features:
    print(f"  - {feat}")

---

## Feature Engineering

The SQLite dump holds raw columns only. I regenerate `log_overhead`, lag features, rolling weather means, and cyclical hour encodings — matching the regression notebook exactly so the saved models can score this data without modification.

In [ ]:
# | label: feature-engineering

df["log_overhead"] = np.log(df["pue"] - 1.0)
df = df[np.isfinite(df["log_overhead"])]

# Cyclical hour
df["hour_sin"] = np.sin(2 * np.pi * df.index.hour / 24)
df["hour_cos"] = np.cos(2 * np.pi * df.index.hour / 24)

# Rolling weather
df["temp_roll_6h"] = df["outdoor_air_temp"].rolling("6h").mean()
df["temp_roll_24h"] = df["outdoor_air_temp"].rolling("24h").mean()

# Lag features on log-overhead target
for lag in [1, 12, 288, 2016]:
    df[f"log_overhead_lag_{lag}"] = df["log_overhead"].shift(lag)

df = df.dropna()
print(f"Post-feature-engineering: {len(df):,} rows")

---

## Classifier Feature Set

Same feature universe as the regression model. Note that `log_overhead_lag_1` correlates ~0.999 with the target — any model will lean on it heavily, which becomes the central object of investigation in the ablation section below.

The split is strictly time-ordered. **Any shuffle-based split would leak the target through the lag features** and inflate every downstream metric.

In [ ]:
# | label: classifier-features

clf_features = [
    "outdoor_air_temp",
    "outdoor_air_humidity",
    "temp_roll_6h",
    "temp_roll_24h",
    "hour_sin",
    "hour_cos",
    "log_overhead_lag_1",
    "log_overhead_lag_12",
    "log_overhead_lag_288",
    "log_overhead_lag_2016",
]

# Time-ordered split (NO shuffling — lag features would leak)
split_idx = int(len(df) * (1 - Config.TEST_SPLIT_RATIO))
X = df[clf_features]
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]

scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Train: {len(X_train):,} rows  |  Test: {len(X_test):,} rows")
print(f"Train period: {X_train.index.min()} → {X_train.index.max()}")
print(f"Test period:  {X_test.index.min()} → {X_test.index.max()}")

---

## Task 1 — PUE Efficiency Tier Classifier

### Data-Driven Thresholds

Standard industry tiers (Excellent < 1.2, Good 1.2–1.5, Poor > 1.5) collapse every row in this dataset into a single "Excellent" class — the facility runs far more efficiently than the industry default assumes. A classifier trained on that label distribution would learn the majority-class shortcut and achieve trivially high accuracy by predicting one class for everything.

Instead, I use **tertiles of the observed PUE distribution**, which produces a balanced three-class problem grounded in this facility's own operating envelope.

In [ ]:
# | label: tier-thresholds

print("PUE percentiles in this dataset:")
for q in [0.10, 0.25, 0.50, 0.75, 0.90, 0.99]:
    print(f"  {int(q * 100):3d}th pct: {df['pue'].quantile(q):.4f}")

t1 = df["pue"].quantile(1 / 3)
t2 = df["pue"].quantile(2 / 3)
print(f"\nTertile cutoffs: t1 = {t1:.4f}, t2 = {t2:.4f}")


def assign_tier(pue):
    if pue <= t1:
        return 0  # Most efficient
    elif pue <= t2:
        return 1  # Typical
    else:
        return 2  # Least efficient


df["tier"] = df["pue"].apply(assign_tier)
tier_labels = ["Most efficient", "Typical", "Least efficient"]

print("\nClass distribution:")
print(df["tier"].value_counts().sort_index())

### Train Two Baseline Models

In [ ]:
# | label: tier-train

y = df["tier"]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Multinomial logistic regression
logreg_tier = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg_tier.fit(X_train_s, y_train)
y_pred_lr = logreg_tier.predict(X_test_s)

print("=== Multinomial Logistic Regression ===")
print(classification_report(y_test, y_pred_lr, target_names=tier_labels))

# Random forest (no scaling needed)
rf_tier = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced",
)
rf_tier.fit(X_train, y_train)
y_pred_rf = rf_tier.predict(X_test)

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, target_names=tier_labels))

### Confusion Matrices

In [ ]:
# | label: tier-confusion

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
short_labels = ["Most", "Typical", "Least"]

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr, ax=axes[0], display_labels=short_labels, cmap="Blues"
)
axes[0].set_title("Logistic Regression")
axes[0].set_xlabel("Predicted class")

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf, ax=axes[1], display_labels=short_labels, cmap="Blues"
)
axes[1].set_title("Random Forest")
axes[1].set_xlabel("Predicted class")

plt.tight_layout()
plt.savefig(Config.PLOT_DIR / "tier_confusion_matrices.png", bbox_inches="tight")
plt.show()

### Feature Importance

The importance plot foreshadows the central finding: short-term lags dominate. The ablation section quantifies exactly how much.

In [ ]:
# | label: tier-importance

imp = pd.Series(rf_tier.feature_importances_, index=clf_features).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
imp.plot(kind="barh", ax=ax, color="#2E86AB")
ax.set_xlabel("Importance")
ax.set_title("Random Forest — feature importance for PUE tier classifier")
plt.tight_layout()
plt.savefig(Config.PLOT_DIR / "tier_feature_importance.png", bbox_inches="tight")
plt.show()

---

## Task 2 — Anomaly / High-Consumption Flag

Binary label: is this 5-minute window in the top decile for **either** cooling load **or** IT load? Positive rate is ~10–19% by construction (the OR of two 90th-percentile indicators).

In [ ]:
# | label: anomaly-label

cooling_p90 = df["cooling_kw"].quantile(0.90)
it_p90 = df["it_power_kw"].quantile(0.90)

print(f"cooling_kw  90th pct: {cooling_p90:.2f} kW")
print(f"it_power_kw 90th pct: {it_p90:.2f} kW")

df["is_anomaly"] = (
    (df["cooling_kw"] > cooling_p90) | (df["it_power_kw"] > it_p90)
).astype(int)

print(f"\nPositive rate: {df['is_anomaly'].mean():.2%}")
print(df["is_anomaly"].value_counts())

### Train the Anomaly Classifier

For imbalanced classification, **PR-AUC is the honest metric**. Accuracy and ROC-AUC both flatter the model on a dataset where "always predict Normal" already wins ~85% of the time.

In [ ]:
# | label: anomaly-train

y_anom = df["is_anomaly"]
y_train_a, y_test_a = y_anom.iloc[:split_idx], y_anom.iloc[split_idx:]

logreg_anom = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg_anom.fit(X_train_s, y_train_a)
y_pred_a = logreg_anom.predict(X_test_s)
y_proba_a = logreg_anom.predict_proba(X_test_s)[:, 1]

print("=== Anomaly Classifier — Logistic Regression ===")
print(classification_report(y_test_a, y_pred_a, target_names=["Normal", "High load"]))
print(f"ROC-AUC: {roc_auc_score(y_test_a, y_proba_a):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test_a, y_proba_a):.4f}")

In [ ]:
# | label: anomaly-confusion

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test_a,
    y_pred_a,
    ax=ax,
    display_labels=["Normal", "High load"],
    cmap="Oranges",
)
ax.set_title("Anomaly classifier")
ax.set_xlabel("Predicted class")
plt.tight_layout()
plt.savefig(Config.PLOT_DIR / "anomaly_confusion.png", bbox_inches="tight")
plt.show()

---

## Stacked Classifier — Reusing the 24h Regression Forecast

The saved 24h regression model already produces a useful estimate of future PUE. Feeding its prediction into the tier classifier is a cheap stacking move: the classifier no longer has to *forecast* PUE, only decide which tier the forecast falls into. This is also a clean demonstration of model composition across notebooks.

In [ ]:
# | label: stacked-classifier

model_24h = loaded_models["24h"]["model"]
scaler_24h = loaded_models["24h"]["scaler"]

# Score the whole dataset with the saved regression model
X_reg = df[regression_features]
X_reg_s = scaler_24h.transform(X_reg)
df["pred_log_overhead_24h"] = model_24h.predict(X_reg_s)
df["pred_pue_24h"] = np.exp(df["pred_log_overhead_24h"]) + 1.0

# Add the forecast as an extra classifier feature
clf_features_stacked = clf_features + ["pred_pue_24h"]
X_stk = df[clf_features_stacked]
X_stk_train, X_stk_test = X_stk.iloc[:split_idx], X_stk.iloc[split_idx:]

sc_stk = RobustScaler()
X_stk_train_s = sc_stk.fit_transform(X_stk_train)
X_stk_test_s = sc_stk.transform(X_stk_test)

logreg_stk = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg_stk.fit(X_stk_train_s, y_train)
y_pred_stk = logreg_stk.predict(X_stk_test_s)

print("=== Tier classifier WITH 24h forecast as feature ===")
print(classification_report(y_test, y_pred_stk, target_names=tier_labels))

---

# The Validation Framework

Headline numbers above look defensible. The four checks below — persistence baselines, walk-forward CV, feature ablation, and operational threshold tuning — were built to expose modes of model self-deception that are easy to miss on time-series data. Each one independently changes how the model should be interpreted.

---

## Rigor Check 1 — Persistence Baseline

Before trusting any model's accuracy on time-series data, compare it to the trivial "predict the same class as the previous observation" baseline. If persistence is competitive, the model isn't adding value — it's measuring autocorrelation.

In [ ]:
# | label: persistence-baseline

# Persistence prediction for tier: whatever tier the previous row was in
y_pers_tier = df["tier"].shift(1).iloc[split_idx:].dropna()
y_test_aligned = y_test.loc[y_pers_tier.index]

acc_pers = (y_pers_tier.values == y_test_aligned.values).mean()
f1_pers = f1_score(y_test_aligned, y_pers_tier, average="macro")

print("=== Persistence baseline (predict tier_{t+1} = tier_t) ===")
print(f"Accuracy: {acc_pers:.4f}")
print(f"Macro F1: {f1_pers:.4f}")

print("\n=== Comparison ===")
acc_lr = (y_pred_lr == y_test).mean()
acc_rf = (y_pred_rf == y_test).mean()
f1_lr = f1_score(y_test, y_pred_lr, average="macro")
f1_rf = f1_score(y_test, y_pred_rf, average="macro")

print(f"{'Model':25s}  {'Accuracy':>10s}  {'Macro F1':>10s}")
print(f"{'Persistence baseline':25s}  {acc_pers:>10.4f}  {f1_pers:>10.4f}")
print(f"{'Logistic regression':25s}  {acc_lr:>10.4f}  {f1_lr:>10.4f}")
print(f"{'Random forest':25s}  {acc_rf:>10.4f}  {f1_rf:>10.4f}")

# Same check for the anomaly label
y_pers_anom = df["is_anomaly"].shift(1).iloc[split_idx:].dropna()
y_anom_aligned = y_test_a.loc[y_pers_anom.index]

print(f"\n=== Anomaly persistence baseline ===")
print(f"Accuracy: {(y_pers_anom.values == y_anom_aligned.values).mean():.4f}")
print(f"F1 (pos): {f1_score(y_anom_aligned, y_pers_anom):.4f}")
print(
    f"(Anomaly LR:  Accuracy {(y_pred_a == y_test_a).mean():.4f}, F1 {f1_score(y_test_a, y_pred_a):.4f})"
)

**What this means.** Persistence (97.2% accuracy) outperforms both trained models (94.1% RF, 89.5% LR) on the tier task. The same pattern holds for the anomaly task — persistence at 99.3% accuracy vs. the trained classifier at 56%. This isn't a modeling failure; it's a property of the data. PUE barely moves over 5 minutes, so the previous tier is nearly always the next tier. The 94% headline reads very differently once you know what the floor is.

---

## Rigor Check 2 — Walk-Forward Cross-Validation

A single train/test split gives one number. Walk-forward gives a *distribution*: what performance looks like across 18 different monthly windows of retraining and out-of-sample evaluation. This is the operational metric — what you'd actually see in production with monthly retraining.

In [ ]:
# | label: walk-forward-classifier


def walk_forward_classifier(
    df_wf,
    feature_cols,
    target_col,
    task="multiclass",
    initial_train_days=180,
    test_days=30,
    step_days=30,
    max_windows=24,
    model_factory=None,
):
    """
    Walk-forward evaluation for classifiers. Matches the regression
    notebook's walk_forward_eval in spirit. For `task="multiclass"` reports
    accuracy and macro F1; for `task="binary"` reports ROC-AUC and PR-AUC.
    """
    if model_factory is None:
        model_factory = lambda: LogisticRegression(
            max_iter=1000, class_weight="balanced"
        )

    results = []
    start = df_wf.index.min()
    end = df_wf.index.max()
    cur_train_end = start + pd.Timedelta(days=initial_train_days)
    window_idx = 0

    while (
        cur_train_end + pd.Timedelta(days=test_days) <= end and window_idx < max_windows
    ):
        test_start = cur_train_end
        test_end = test_start + pd.Timedelta(days=test_days)

        train = df_wf[df_wf.index < cur_train_end]
        test = df_wf[(df_wf.index >= test_start) & (df_wf.index < test_end)]

        if len(train) < 1000 or len(test) < 100:
            cur_train_end += pd.Timedelta(days=step_days)
            window_idx += 1
            continue

        X_tr, y_tr = train[feature_cols], train[target_col]
        X_te, y_te = test[feature_cols], test[target_col]

        sc = RobustScaler()
        X_tr_s = sc.fit_transform(X_tr)
        X_te_s = sc.transform(X_te)

        m = model_factory()
        m.fit(X_tr_s, y_tr)
        pred = m.predict(X_te_s)

        row = {
            "test_start": test_start,
            "train_size": len(train),
            "accuracy": (pred == y_te).mean(),
        }

        if task == "multiclass":
            row["macro_f1"] = f1_score(y_te, pred, average="macro")
            row["weighted_f1"] = f1_score(y_te, pred, average="weighted")
        else:  # binary
            proba = m.predict_proba(X_te_s)[:, 1]
            # Guard against single-class windows
            if len(np.unique(y_te)) > 1:
                row["roc_auc"] = roc_auc_score(y_te, proba)
                row["pr_auc"] = average_precision_score(y_te, proba)
            else:
                row["roc_auc"] = np.nan
                row["pr_auc"] = np.nan
            row["f1"] = f1_score(y_te, pred)

        results.append(row)
        cur_train_end += pd.Timedelta(days=step_days)
        window_idx += 1

    return pd.DataFrame(results)


# Tier classifier walk-forward
wf_tier = walk_forward_classifier(df, clf_features, "tier", task="multiclass")
print(f"Tier classifier — {len(wf_tier)} windows\n")
print(wf_tier[["test_start", "accuracy", "macro_f1"]].to_string(index=False))
print(f"\nMedian accuracy: {wf_tier['accuracy'].median():.4f}")
print(f"Median macro-F1: {wf_tier['macro_f1'].median():.4f}")
print(
    f"Macro-F1 IQR:    "
    f"{wf_tier['macro_f1'].quantile(0.25):.4f} – "
    f"{wf_tier['macro_f1'].quantile(0.75):.4f}"
)

In [ ]:
# | label: walk-forward-anomaly

wf_anom = walk_forward_classifier(df, clf_features, "is_anomaly", task="binary")
print(f"Anomaly classifier — {len(wf_anom)} windows\n")
print(wf_anom[["test_start", "roc_auc", "pr_auc", "f1"]].to_string(index=False))
print(f"\nMedian ROC-AUC: {wf_anom['roc_auc'].median():.4f}")
print(f"Median PR-AUC:  {wf_anom['pr_auc'].median():.4f}")

In [ ]:
# | label: walk-forward-plot

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].bar(
    wf_tier["test_start"],
    wf_tier["macro_f1"],
    width=25,
    color="#2A9D8F",
    alpha=0.75,
)
axes[0].axhline(
    wf_tier["macro_f1"].median(),
    color="blue",
    linestyle=":",
    label=f"Median = {wf_tier['macro_f1'].median():.3f}",
)
axes[0].set_ylabel("Macro F1")
axes[0].set_title("Walk-forward: tier classifier (30-day windows)")
axes[0].legend()

axes[1].bar(
    wf_anom["test_start"],
    wf_anom["pr_auc"],
    width=25,
    color="#E76F51",
    alpha=0.75,
)
axes[1].axhline(
    wf_anom["pr_auc"].median(),
    color="blue",
    linestyle=":",
    label=f"Median = {wf_anom['pr_auc'].median():.3f}",
)
axes[1].set_ylabel("PR-AUC")
axes[1].set_xlabel("Test window start")
axes[1].set_title("Walk-forward: anomaly classifier (30-day windows)")
axes[1].legend()

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(Config.PLOT_DIR / "walk_forward_classifiers.png", bbox_inches="tight")
plt.show()

**What this means.** Tier-classifier macro-F1 ranges from 0.18 to 1.00 across windows, with a median of 0.84 and an IQR of 0.57–0.99. The single-split number landed on a favorable window. Reporting only that point estimate would have overstated operational performance considerably. The wide IQR is itself a signal — in production this would warrant a drift detector and a retraining cadence shorter than the variance horizon.

---

## Rigor Check 3 — Lag Feature Ablation

The feature importance plot showed `log_overhead_lag_1` and `log_overhead_lag_12` dominating. This ablation quantifies how much real signal exists beyond autocorrelation by retraining with progressively fewer lag features.

In [ ]:
# | label: lag-ablation

ablation_sets = {
    "A: all features (baseline)": clf_features,
    "B: drop lag_1": [c for c in clf_features if c != "log_overhead_lag_1"],
    "C: drop lag_1 + lag_12": [
        c
        for c in clf_features
        if c not in ("log_overhead_lag_1", "log_overhead_lag_12")
    ],
    "D: weather + time only (no lags)": [
        "outdoor_air_temp",
        "outdoor_air_humidity",
        "temp_roll_6h",
        "temp_roll_24h",
        "hour_sin",
        "hour_cos",
    ],
}

ablation_rows = []
for label, feats in ablation_sets.items():
    X_ab_tr = df[feats].iloc[:split_idx]
    X_ab_te = df[feats].iloc[split_idx:]

    sc = RobustScaler()
    X_ab_tr_s = sc.fit_transform(X_ab_tr)
    X_ab_te_s = sc.transform(X_ab_te)

    m = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced",
    )
    m.fit(X_ab_tr_s, y_train)
    pred = m.predict(X_ab_te_s)

    ablation_rows.append(
        {
            "config": label,
            "n_features": len(feats),
            "accuracy": (pred == y_test).mean(),
            "macro_f1": f1_score(y_test, pred, average="macro"),
        }
    )

ablation_df = pd.DataFrame(ablation_rows)
print("=== Lag ablation (Random Forest, tier classifier) ===\n")
print(ablation_df.to_string(index=False))

print(
    f"\nΔ from dropping lag_1:       "
    f"{ablation_df.iloc[1]['macro_f1'] - ablation_df.iloc[0]['macro_f1']:+.4f} macro-F1"
)
print(
    f"Δ from dropping lag_1+lag_12: "
    f"{ablation_df.iloc[2]['macro_f1'] - ablation_df.iloc[0]['macro_f1']:+.4f} macro-F1"
)
print(
    f"Δ dropping all lags:         "
    f"{ablation_df.iloc[3]['macro_f1'] - ablation_df.iloc[0]['macro_f1']:+.4f} macro-F1"
)

**What this means.** The collapse from 0.92 → 0.30 macro-F1 is the cleanest finding in the project: the model wasn't learning facility behavior, it was extrapolating the recent past. The 24h and weekly lags carried a small but real ~0.10 F1 above the random-guess floor — the daily cycle is the only honest predictive signal in the feature set. The 5-minute continuity is not predictive in any meaningful sense; it's the dataset's own autocorrelation reflected back. Configuration D is the version of this model that could actually be used to reason about facility behavior, rather than just extrapolate the last few minutes.

---

## Rigor Check 4 — Anomaly Threshold Tuning

The anomaly classifier with `class_weight="balanced"` produced 99% recall but 34% precision — catching almost every high-load event at the cost of 17K false alarms. Unusable for any real ops workflow. Tuning the decision threshold from the precision-recall curve produces a much better operating point.

In [ ]:
# | label: threshold-tuning

precisions, recalls, thresholds = precision_recall_curve(y_test_a, y_proba_a)

# F1 at each threshold. precision_recall_curve returns len(thresholds)+1
# precision/recall values, so align them.
f1_scores = (
    2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-12)
)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Default threshold (0.50):")
print(f"  Precision: {(y_pred_a[y_test_a == 1] == 1).mean():.4f}")
print(f"  Recall:    {y_pred_a[y_test_a == 1].mean():.4f}")
print(f"  F1:        {f1_score(y_test_a, y_pred_a):.4f}")

print(f"\nF1-optimal threshold ({best_threshold:.4f}):")
print(f"  Precision: {precisions[best_idx]:.4f}")
print(f"  Recall:    {recalls[best_idx]:.4f}")
print(f"  F1:        {f1_scores[best_idx]:.4f}")

# Show the tradeoff curve
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(recalls, precisions, color="#E76F51", linewidth=2)
axes[0].scatter(
    recalls[best_idx],
    precisions[best_idx],
    color="blue",
    s=80,
    zorder=5,
    label=f"F1-optimal @ t={best_threshold:.3f}",
)
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title(
    f"PR curve (PR-AUC = {average_precision_score(y_test_a, y_proba_a):.3f})"
)
axes[0].legend()
axes[0].grid(alpha=0.3)

# F1 vs threshold
axes[1].plot(thresholds, f1_scores, color="#2A9D8F", linewidth=2)
axes[1].axvline(
    best_threshold, color="blue", linestyle=":", label=f"Best t = {best_threshold:.3f}"
)
axes[1].axvline(0.5, color="gray", linestyle="--", alpha=0.5, label="Default t = 0.5")
axes[1].set_xlabel("Decision threshold")
axes[1].set_ylabel("F1 score")
axes[1].set_title("F1 as a function of threshold")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(Config.PLOT_DIR / "anomaly_threshold_tuning.png", bbox_inches="tight")
plt.show()

# Report the tuned confusion matrix
y_pred_tuned = (y_proba_a >= best_threshold).astype(int)
print("\n=== Confusion matrix at F1-optimal threshold ===")
print(
    classification_report(y_test_a, y_pred_tuned, target_names=["Normal", "High load"])
)

# Comparison plot: default vs tuned threshold
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test_a,
    y_pred_a,
    ax=axes[0],
    display_labels=["Normal", "High load"],
    cmap="Oranges",
)
axes[0].set_title(f"Default threshold (t=0.50)\nFP: {17477}, FN: {90}")

ConfusionMatrixDisplay.from_predictions(
    y_test_a,
    y_pred_tuned,
    ax=axes[1],
    display_labels=["Normal", "High load"],
    cmap="Oranges",
)
axes[1].set_title(f"F1-optimal threshold (t={best_threshold:.3f})")

plt.tight_layout()
plt.savefig(Config.PLOT_DIR / "anomaly_threshold_comparison.png", bbox_inches="tight")
plt.show()

# Summary of improvement
fp_default = 17477
fn_default = 90
fp_tuned = y_pred_tuned[(y_test_a == 0).values].sum()
fn_tuned = y_pred_tuned[(y_test_a == 1).values & (y_pred_tuned == 0)].sum()
f1_default = f1_score(y_test_a, y_pred_a)
f1_tuned = f1_score(y_test_a, y_pred_tuned)

print("\n=== Threshold tuning impact ===")
print(
    f"False positives: {fp_default:,} → {fp_tuned:,} (reduced by {(fp_default-fp_tuned)/fp_default*100:.1f}%)"
)
print(
    f"False negatives: {fn_default:,} → {fn_tuned:,} (changed by {(fn_tuned-fn_default)/fn_default*100:+.1f}%)"
)
print(
    f"F1 score:        {f1_default:.4f} → {f1_tuned:.4f} ({(f1_tuned-f1_default)*100:+.1f} pp)"
)

**What this means.** Moving the threshold from 0.50 to 0.68 cut false positives by 71.6% (17,477 → 4,957), eliminated all false negatives (90 → 0), and raised F1 from 0.51 to 0.66. Which threshold to choose in production depends on operational cost — F1-optimal is the default if false positives and false negatives carry the same cost; slide left if missed events are catastrophic, right if alert fatigue dominates. The PR curve makes that tradeoff explicit rather than inheriting it implicitly from a scikit-learn default.

---

## Save the Classifiers

In [ ]:
# | label: save-models

artifacts = [
    (
        "tier_logreg",
        {
            "model": logreg_tier,
            "scaler": scaler,
            "features": clf_features,
            "thresholds": (t1, t2),
            "labels": tier_labels,
        },
    ),
    (
        "tier_rf",
        {
            "model": rf_tier,
            "features": clf_features,
            "thresholds": (t1, t2),
            "labels": tier_labels,
        },
    ),
    (
        "anomaly_logreg",
        {
            "model": logreg_anom,
            "scaler": scaler,
            "features": clf_features,
            "cooling_p90": cooling_p90,
            "it_p90": it_p90,
        },
    ),
    (
        "tier_stacked_logreg",
        {
            "model": logreg_stk,
            "scaler": sc_stk,
            "features": clf_features_stacked,
            "thresholds": (t1, t2),
            "labels": tier_labels,
        },
    ),
]

for name, obj in artifacts:
    path = Config.MODEL_DIR / f"{name}.pkl"
    try:
        with open(path, "wb") as f:
            pickle.dump(obj, f)
        print(f"✓ Saved {path}")
    except (pickle.PickleError, IOError) as e:
        print(f"✗ Failed to save {name}: {e}")

---

*04-14-2026 · Python 3 · scikit-learn · pandas*